In [ ]:
# SJSU Student Resource Navigator
### AI for Social Good | Fundamentals of MIS | Spring 2026

#This notebook demonstrates a text generation system that helps SJSU students
#in financial distress identify which campus resources apply to their situation
#and what to do next. The system uses Google's Gemini 2.5 Flash model.

#**SDG Alignment:** SDG 1 (No Poverty), SDG 4 (Quality Education)
#**AI Capability:** Text Generation (Lab 1)
#**Community Scope:** San Jose State University students facing financial hardship

In [ ]:
# Install the Google Generative AI library
!pip install -q google-generativeai

In [ ]:
import google.generativeai as genai
from google.colab import userdata
import time

# Initialize Gemini with your API key stored in Colab Secrets
# Key icon in left sidebar > add secret named GEMINI_API_KEY > toggle notebook access ON
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print("Gemini initialized successfully.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini initialized successfully.


In [ ]:
## Part 1: The System Prompt

#The system prompt is the policy layer of this tool. It defines:
#- Who the AI thinks it is talking to
#- What resources it knows about
#- What format it responds in
#- What it will and will not do

#Changing this prompt changes who the tool can serve and what it tells them.
#This is a design decision, not just a technical setting.

In [ ]:
# SYSTEM PROMPT — This is the core policy decision of the tool.
# It tells Gemini what role to play, what it knows, and how to respond.
# Every word here affects who gets helped and how accurately.

sjsu_system_prompt = """
You are a student resource navigator for San Jose State University (SJSU).
Your job is to help SJSU students who are experiencing financial hardship
identify which campus resources apply to their specific situation and what
their next step should be.

You have knowledge of the following SJSU resources:
- Emergency Financial Assistance Fund: one-time grants up to $500 for
  students with sudden financial emergencies. Apply through Student Financial
  Services (SFS), located in Student Services Center Room 201.
- Spartan Food Pantry: free groceries available to all enrolled SJSU students,
  no income requirement. Located in the Student Wellness Center, open Monday
  through Friday 10am-4pm.
- Fee Deferral Program: allows students to defer semester fees up to 60 days
  with a signed agreement. Contact SFS directly.
- Financial Aid Appeal: students whose financial situation has changed since
  submitting their FAFSA can appeal for additional aid. Submit through
  MySJSU portal under Financial Aid > Appeals.
- Spartan Shops Food Access Program: discounted meal plans for students
  demonstrating financial need. Apply at the Spartan Shops office.
- Basic Needs Center: connects students to housing support, CalFresh
  enrollment assistance, and emergency housing. Located in Clark Hall 126.
- SJSU Cares: case management for students in crisis. Connects students
  across multiple departments. Email sjsucares@sjsu.edu.

When a student describes their situation:
1. Identify which 1-3 resources are most relevant to their specific situation.
2. List them in order of urgency — what should they do FIRST.
3. For each resource, give the specific action: where to go, who to contact,
   what to bring.
4. Keep your response under 200 words.
5. End with one sentence acknowledging the difficulty of their situation.
6. Do NOT recommend resources that clearly do not apply.
7. If the situation is unclear, ask one clarifying question instead of guessing.
"""

print("System prompt loaded.")
print(f"Prompt length: {len(sjsu_system_prompt)} characters")

System prompt loaded.
Prompt length: 1962 characters


In [ ]:
# Core function: sends a student's message to Gemini and returns a response
# The system prompt stays constant; only the student's input changes each call.

def get_resource_recommendation(student_message, system_prompt=sjsu_system_prompt):
    """
    Takes a student's description of their situation and returns
    a plain-language recommendation of which SJSU resources apply
    and what to do first.

    Parameters:
        student_message (str): The student's description of their situation
        system_prompt (str): The policy layer defining what the AI knows

    Returns:
        str: Gemini's recommendation
    """
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=system_prompt
    )

    response = model.generate_content(student_message)

    # Rate limit buffer for free tier (5 requests/min)
    time.sleep(12)

    return response.text


print("Function defined. Ready to run.")

Function defined. Ready to run.


In [ ]:
## Part 2: Test Cases

#We test the system against four realistic student scenarios.
#These represent the range of students who would actually use this tool.

In [ ]:
# TEST CASE 1: Financial hold blocking registration
# Represents the most common crisis scenario at SJSU

test_1 = """
I have a financial hold on my account and I can't register for fall classes.
I work part time at Target, about 20 hours a week, and my family income is
under $40,000 a year. I already submitted my FAFSA but my aid hasn't come
through yet. I don't know what to do. Registration closes in two weeks.
"""

print("=== TEST CASE 1: Financial Hold ===")
print(f"Input: {test_1.strip()}")
print("\n--- Gemini Response ---")
response_1 = get_resource_recommendation(test_1)
print(response_1)

=== TEST CASE 1: Financial Hold ===
Input: I have a financial hold on my account and I can't register for fall classes.
I work part time at Target, about 20 hours a week, and my family income is
under $40,000 a year. I already submitted my FAFSA but my aid hasn't come
through yet. I don't know what to do. Registration closes in two weeks.

--- Gemini Response ---
It sounds incredibly stressful to have a financial hold preventing registration, especially with the deadline approaching.

Your first step should be to address the financial hold directly:

1.  **Fee Deferral Program:** This allows you to defer semester fees for up to 60 days by signing an agreement. This is crucial to get your hold lifted so you can register.
    *   **Action:** Contact **Student Financial Services (SFS)** directly. They are located in the Student Services Center Room 201.
2.  **Financial Aid Appeal:** Given your family income is under $40,000 and your aid hasn't come through yet, or if your financial situat

In [ ]:
# TEST CASE 2: Food insecurity
# Student may not know the Spartan Food Pantry exists or that they qualify

test_2 = """
I've been skipping meals because I ran out of money before the end of the month.
I'm embarrassed to ask for help but I don't know what else to do. I'm a
sophomore, enrolled full time. Is there anything at SJSU that can help me
with food? I don't want to have to drop classes to work more hours.
"""

print("=== TEST CASE 2: Food Insecurity ===")
print(f"Input: {test_2.strip()}")
print("\n--- Gemini Response ---")
response_2 = get_resource_recommendation(test_2)
print(response_2)

=== TEST CASE 2: Food Insecurity ===
Input: I've been skipping meals because I ran out of money before the end of the month.
I'm embarrassed to ask for help but I don't know what else to do. I'm a
sophomore, enrolled full time. Is there anything at SJSU that can help me
with food? I don't want to have to drop classes to work more hours.

--- Gemini Response ---
I understand how incredibly difficult it must be to be skipping meals. You're not alone, and SJSU has resources to help immediately:

1.  **Spartan Food Pantry:** For immediate free groceries, visit the Student Wellness Center, open Monday-Friday, 10 am-4 pm. Just bring your Tower ID; no income requirement.
2.  **Emergency Financial Assistance Fund:** You may qualify for a one-time grant up to $500 to help with sudden financial emergencies. Apply through Student Financial Services (SFS) in Student Services Center Room 201.
3.  **Basic Needs Center:** They can assist you with enrolling in CalFresh, a federal program that provides

In [ ]:
# TEST CASE 3: Multiple overlapping crises
# Student has housing, food, and financial issues simultaneously.
# This tests whether the system can prioritize correctly.

test_3 = """
I'm a first-generation college student. My mom just lost her job and we
might lose our apartment next month. I also have a balance on my student
account from last semester. I don't know where to start — there's too much
going on at once. I need help with housing, food, and my account balance.
"""

print("=== TEST CASE 3: Overlapping Crises ===")
print(f"Input: {test_3.strip()}")
print("\n--- Gemini Response ---")
response_3 = get_resource_recommendation(test_3)
print(response_3)

=== TEST CASE 3: Overlapping Crises ===
Input: I'm a first-generation college student. My mom just lost her job and we
might lose our apartment next month. I also have a balance on my student
account from last semester. I don't know where to start — there's too much
going on at once. I need help with housing, food, and my account balance.

--- Gemini Response ---


TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 27.717586003s.

In [ ]:
# TEST CASE 4: Changed financial circumstances mid-year
# Student's situation changed after FAFSA was submitted.
# Tests whether the system knows about the appeal process.

test_4 = """
My financial situation changed a lot since I submitted my FAFSA last year.
My dad was laid off in January and now my parents can't help me pay for
school at all. My aid package was based on their old income. Can I get
more help?
"""

print("=== TEST CASE 4: Changed Financial Circumstances ===")
print(f"Input: {test_4.strip()}")
print("\n--- Gemini Response ---")
response_4 = get_resource_recommendation(test_4)
print(response_4)

In [ ]:
## Part 3: Edge Case Elicitation

Before analyzing ethics, we deliberately attempt to break the system.
We design a prompt targeting a user whose situation falls outside the
assumed majority and a condition that makes the AI's confidence misleading.

This is required for Part 4 of the assignment.

In [ ]:
# EDGE CASE: Vague input in Spanish from a non-English-speaking student
# This targets two failure conditions simultaneously:
# (1) a user outside the assumed majority (non-English speaker)
# (2) a vague input that fits multiple categories without clear direction
#
# The system prompt is written entirely in English.
# We test whether Gemini can still route correctly — or whether it fails.

edge_case = """
Hola, necesito ayuda. No tengo dinero y no sé qué hacer. Estoy en la universidad
pero no entiendo los recursos. ¿Puede ayudarme?
"""

# Translation for reference:
# "Hello, I need help. I have no money and I don't know what to do.
# I am at the university but I don't understand the resources. Can you help me?"

print("=== EDGE CASE: Vague Input in Spanish ===")
print(f"Input: {edge_case.strip()}")
print("\nTranslation: 'Hello, I need help. I have no money and I don't know")
print("what to do. I am at the university but I don't understand the resources.'")
print("\n--- Gemini Response ---")
response_edge = get_resource_recommendation(edge_case)
print(response_edge)

In [ ]:
# ASSESSMENT OF EDGE CASE OUTPUT
# Document this for Part 4 of the assignment.

print("=== EDGE CASE ASSESSMENT ===")
print("""
Prompt used: Vague financial distress message in Spanish, no specific
detail about what kind of help is needed.

Output from Gemini: [paste or describe what Gemini actually returned above]

One-sentence assessment: [Choose one after you see the output]

OPTION A (if Gemini responded in English only):
Near-miss — the system responded but defaulted to English, excluding the
student from understanding the recommendation despite detecting the language.

OPTION B (if Gemini responded in Spanish):
Acceptable — the system detected the language and responded appropriately,
but the vagueness of the input may have led to a generic rather than
targeted recommendation.

OPTION C (if Gemini hallucinated resources or gave wrong information):
Failure — the system returned confident but incorrect resource information,
which could send a student to the wrong office at a critical moment.
""")

In [ ]:
## Part 4: Ethics and Edge Cases

### 4.1 One Failure Case

#**Concrete input:** A Spanish-speaking student in financial distress sends
#a vague message with no specific detail about their situation type.

#**What the AI returned:** [Reference your actual edge case output above]

#**Real-world consequence:** If the system responds only in English, the
#student — already under financial stress and potentially unfamiliar with
#university systems — cannot understand the recommendation. They miss the
#Fee Deferral Program deadline, their hold remains, and they cannot register
#for the following semester. The student most in need of help is the one
#the system fails.

#**Lab connection:** Lab 2 showed that the AI's routing decisions depend
#entirely on the categories built into the schema. When the input doesn't
#cleanly fit a category — as with a vague, emotionally distressed message —
#the system either guesses or returns a generic response. The same dynamic
#applies here: vague input produces unreliable output regardless of how
#well the system prompt is written.

#---

### 4.2 Oversight Decision

#**Position:** A peer advisor in the Basic Needs Center reviews any
#response before it is sent to a student who wrote in a language other
#than English or whose input contained no specific resource category.

#**Justification:** Lab 1 showed that a single line in the system prompt
#determines who the tool can serve. When that line is absent or
#insufficient — as it is for non-English inputs — the output is
#unreliable enough that a human must catch it before it reaches the student.

#---

### 4.3 The One Change

#**Change:** Add a language detection instruction to the system prompt
#that directs Gemini to respond in the same language the student used,
#and to ask one clarifying question if the situation is too vague to
#route accurately.

#**What it costs:** This reduces automation speed. Every vague or
#non-English input now triggers either a clarifying exchange (adding
#one round-trip delay) or a human review queue (adding 4-24 hours
#depending on staffing). For students in acute crisis, that delay
#is not trivial. The tradeoff is accuracy versus immediacy.

In [ ]:
# UPDATED SYSTEM PROMPT — implementing the one change from Part 4.3
# Added: language detection instruction
# Added: clarifying question protocol for vague inputs
# This demonstrates the fix and its effect on the edge case.

sjsu_system_prompt_v2 = """
You are a student resource navigator for San Jose State University (SJSU).
Your job is to help SJSU students who are experiencing financial hardship
identify which campus resources apply to their specific situation and what
their next step should be.

IMPORTANT: Detect the language the student wrote in and respond in that
same language. If the student wrote in Spanish, respond in Spanish.
If the student wrote in Vietnamese, respond in Vietnamese. Always match
the student's language.

If the student's message is too vague to identify which resource applies,
ask ONE clarifying question before making a recommendation. Do not guess.

You have knowledge of the following SJSU resources:
- Emergency Financial Assistance Fund: one-time grants up to $500 for
  students with sudden financial emergencies. Apply through Student Financial
  Services (SFS), located in Student Services Center Room 201.
- Spartan Food Pantry: free groceries available to all enrolled SJSU students,
  no income requirement. Located in the Student Wellness Center,
  open Monday through Friday 10am-4pm.
- Fee Deferral Program: allows students to defer semester fees up to 60 days
  with a signed agreement. Contact SFS directly.
- Financial Aid Appeal: students whose financial situation has changed since
  submitting their FAFSA can appeal for additional aid. Submit through
  MySJSU portal under Financial Aid > Appeals.
- Spartan Shops Food Access Program: discounted meal plans for students
  demonstrating financial need. Apply at the Spartan Shops office.
- Basic Needs Center: connects students to housing support, CalFresh
  enrollment assistance, and emergency housing. Located in Clark Hall 126.
- SJSU Cares: case management for students in crisis. Connects students
  across multiple departments. Email sjsucares@sjsu.edu.

When a student describes their situation:
1. Identify which 1-3 resources are most relevant to their specific situation.
2. List them in order of urgency — what should they do FIRST.
3. For each resource, give the specific action: where to go, who to contact,
   what to bring.
4. Keep your response under 200 words.
5. End with one sentence acknowledging the difficulty of their situation.
6. Do NOT recommend resources that clearly do not apply.
7. If the situation is unclear, ask one clarifying question instead of guessing.
"""

print("=== RE-RUNNING EDGE CASE WITH UPDATED SYSTEM PROMPT ===")
print(f"Input: {edge_case.strip()}")
print("\n--- Updated Gemini Response ---")

model_v2 = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    system_instruction=sjsu_system_prompt_v2
)

response_edge_v2 = model_v2.generate_content(edge_case)
time.sleep(12)
print(response_edge_v2.text)
print("\n--- Comparison ---")
print("Original response (English-only prompt): see Cell 13 output")
print("Updated response (multilingual prompt): see above")
print("The update demonstrates the fix and its tradeoff.")